# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook guides users through loading, exploring, and analyzing the *A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa* using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
# Access metadata attributes directly
print(f"{metadata['name']}: {metadata['description']}")

# Preview other metadata fields
print("Authors (@id):", [author['@id'] for author in metadata.get('author', [])])
print("Record Sets (@id):", metadata.get('recordSet', []))

## 2. Data Overview
Review available record sets, fields, and their IDs.

The record sets define the main tables of data. Each field and column is referenced by its unique `@id`.

Let's inspect the available record sets.

In [ ]:
# Print available record sets and their fields (@id)
record_sets = metadata.get('recordSet', [])
if not record_sets:
    print("No record sets found in metadata.")
else:
    for rs in record_sets:
        if isinstance(rs, dict):
            rs_id = rs.get('@id', '(missing)')
            fields = rs.get('field', [])
        else:
            rs_id = rs
            fields = []
        print(f"Record Set @id: {rs_id}")
        if fields:
            print("  Fields (@id):")
            for f in fields:
                if isinstance(f, dict):
                    print(f"    {f.get('@id', '(missing)')}")
                else:
                    print(f"    {f}")
        else:
            print("  No fields listed.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Always reference by `@id`.

If the `recordSet` list is empty in metadata, we'll check for known record set `@id`s from the schema or documentation. Otherwise, use available IDs from previous step.

In [ ]:
# Try to extract record set IDs from metadata
if not record_sets:
    # Known record set ID from FAIR² Croissant schema
    # For demonstration, we use a commonly structured ID (adjust as needed for your dataset)
    example_record_set_id = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json#kilifi_mental_health_survey'
    record_sets_ids = [example_record_set_id]
else:
    record_sets_ids = []
    for rs in record_sets:
        if isinstance(rs, dict):
            record_sets_ids.append(rs.get('@id'))
        else:
            record_sets_ids.append(rs)

# Load data from each record set
dataframes = {}
for rs_id in record_sets_ids:
    # Use mlcroissant .records() with record_set=@id
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"---\nLoaded DataFrame for Record Set @id: {rs_id}")
    print("Columns:", df.columns.tolist())
    print(df.head(3))

# Select first record set for detailed analysis
main_record_set_id = record_sets_ids[0]
main_df = dataframes[main_record_set_id]

## 4. Exploratory Data Analysis (EDA)
Apply processing steps such as filtering, normalizing numeric fields, and grouping records by a key attribute.

**Note:** All columns referenced by their `@id`. Adjust field/column IDs if schema uses alternative structures.

In [ ]:
# Select a numeric field for analysis by @id
# We'll use GAD-7 score, which is commonly in mental health surveys
numeric_field_id = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json#gad7_score'

# Choose a threshold for filtering
threshold = 10

# Filter records
if numeric_field_id in main_df.columns:
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by 'level_of_education' field (example)
    group_field_id = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json#level_of_education'
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print(f"{numeric_field_id} not found in main dataframe columns. Available columns:")
    print(main_df.columns.tolist())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of GAD-7 scores and visualize mean scores by education level.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of GAD-7 scores
if numeric_field_id in main_df.columns and main_df[numeric_field_id].notna().sum() > 0:
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[numeric_field_id], bins=10, kde=True)
    plt.title("Distribution of GAD-7 Scores")
    plt.xlabel("GAD-7 Score")
    plt.ylabel("Count")
    plt.show()

    # Plot mean GAD-7 by education level
    if group_field_id in main_df.columns:
        group_means = main_df.groupby(group_field_id)[numeric_field_id].mean().dropna()
        plt.figure(figsize=(8,4))
        group_means.plot(kind='bar')
        plt.title("Mean GAD-7 Score by Level of Education")
        plt.xlabel("Level of Education (@id)")
        plt.ylabel("Mean GAD-7 Score")
        plt.tight_layout()
        plt.show()
else:
    print("Cannot plot distribution: GAD-7 field not found or all values missing.")

## 6. Conclusion
Summarize key findings and observations from dataset exploration.

- The FAIR² Mental Health Survey Dataset provides valuable insights on psychological symptom scores and demographics among Kilifi County residents.
- By referencing data entities by `@id`, we ensure that data extraction, filtering, and aggregation are reproducible and robust.
- Initial analysis reveals score distributions and potential demographic trends in survey responses.
- This approach demonstrates how the `mlcroissant` library and AI-Ready data standards support transparent, reproducible analytics workflows for public health datasets.